# Manejo de Memoria, Punteros y Garbage Collector en Go

Go es un lenguaje con gestión automática de memoria, pero a diferencia de otros lenguajes de alto nivel, ofrece control explícito sobre la memoria mediante punteros y permite entender cómo se distribuyen los datos entre el **Stack** y el **Heap**.

## 1. Punteros en Go

Un puntero almacena la dirección de memoria de un valor. A diferencia de C, Go no permite la aritmética de punteros (sumar o restar a una dirección) por razones de seguridad.

In [ ]:
package main

import "fmt"

func main() {
    i := 42
    p := &i         // & genera un puntero a 'i'

    fmt.Println(p)  // Imprime la dirección de memoria
    fmt.Println(*p) // * lee el valor a través del puntero (dereferenciación)

    *p = 21         // Modifica 'i' a través del puntero
    fmt.Println(i)
}

## 2. Stack vs Heap y Escape Analysis

En Go, el compilador decide automáticamente si una variable vive en el **Stack** (rápido, se libera al terminar la función) o en el **Heap** (más lento, gestionado por el GC).

- **Stack**: Para variables locales cuyo tamaño es conocido y no se referencian fuera de la función.
- **Heap**: Para datos que "escapan" al ámbito de la función o tienen un tamaño dinámico.

### Escape Analysis
Es el proceso mediante el cual el compilador decide la ubicación. Si retornas un puntero a una variable local, Go la moverá automáticamente al Heap.

In [ ]:
func crearUsuario() *string {
    nombre := "Maxi" 
    return &nombre // 'nombre' escapa al heap porque se retorna su dirección
}

// Para verificar el escape analysis:
// go build -gcflags="-m" main.go

## 3. Asignación de Memoria: `new` vs `make`

Es vital entender la diferencia entre estas dos funciones integradas.

In [ ]:
// 1. new(T)
// Reserva memoria puesta a cero para un tipo T y retorna un puntero (*T).
p := new(int)
fmt.Printf("Tipo: %T, Valor: %v\n", p, *p) // *int, 0

// 2. make(T, args)
// Solo para slices, maps y channels.
// No retorna un puntero, sino un valor inicializado (con sus estructuras internas listas).
s := make([]int, 10, 20) // slice de len 10 y cap 20

## 4. El Garbage Collector (GC)

Go utiliza un recolector de basura de tipo **Tristate Mark-and-Sweep Concurrent Collector**.

### Características:
- **No compactante**: No mueve los objetos en memoria para evitar pausas largas.
- **Latencia baja**: Prioriza pausas de milisegundos (STW - Stop The World mínimas).
- **Pacing**: Ajusta su velocidad según la tasa de asignación de memoria.

### Control del GC:
Puedes influir en el comportamiento mediante variables de entorno:
- `GOGC`: Define el porcentaje de crecimiento de memoria antes del siguiente ciclo (default 100).
- `GOMEMLIMIT`: (Desde Go 1.19) Define un límite suave de memoria para evitar errores Out Of Memory (OOM).

In [ ]:
import "runtime"
import "runtime/debug"

// Forzar el GC manualmente (raramente necesario)
runtime.GC()

// Ajustar GOGC dinámicamente
debug.SetGCPercent(50)

## 5. Value vs Pointer Semantics

La decisión de pasar por valor o por puntero afecta el rendimiento y el comportamiento.

In [ ]:
type Grande struct {
    datos [1024]int
}

// Semántica de Valor: Copia toda la estructura (Lenta si es grande)
func procesarValor(g Grande) {}

// Semántica de Puntero: Comparte la dirección (Rápida, permite mutabilidad)
func procesarPuntero(g *Grande) {}

## 6. Zero Values y Nil

En Go, las variables no inicializadas tienen un "Zero Value":
- `int`: `0`
- `bool`: `false`
- `string`: `""`
- `punteros`, `slices`, `maps`, `channels`: `nil`